In [1]:
from safetensors.torch import load_file
import json

# Load the weights
weights = load_file(r"D:\Projects\IMPOLS\pretrained_models\gpt2\model.safetensors")

In [2]:
#load config
with open(r"D:\Projects\IMPOLS\pretrained_models\gpt2\config.json") as f:
    hf_config = json.load(f)

# print(hf_config)

In [3]:
from safetensors.torch import load_file

hf_weights = load_file(r"D:\Projects\IMPOLS\pretrained_models\gpt2\model.safetensors")

def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch → Left: {left.shape}, Right: {right.shape}")
    return nn.Parameter(right.clone().detach().float())

def load_hf_into_my_gpt(model, w):
    model.token_emb.weight = assign(model.token_emb.weight, w["wte.weight"])
    model.pos_emb.weight   = assign(model.pos_emb.weight,   w["wpe.weight"])

    for b in range(cfg["n_layer"]):
        p  = f"h.{b}"
        tb = model.tranformer_block[b]

        # LayerNorms
        tb.layernorm1.scale = assign(tb.layernorm1.scale, w[f"{p}.ln_1.weight"])
        tb.layernorm1.shift = assign(tb.layernorm1.shift, w[f"{p}.ln_1.bias"])
        tb.layernorm2.scale = assign(tb.layernorm2.scale, w[f"{p}.ln_2.weight"])
        tb.layernorm2.shift = assign(tb.layernorm2.shift, w[f"{p}.ln_2.bias"])

        # Q, K, V (split from concatenated HuggingFace format)
        qkv_w = w[f"{p}.attn.c_attn.weight"]  # [768, 2304]
        qkv_b = w[f"{p}.attn.c_attn.bias"]    # [2304]
        q_w, k_w, v_w = torch.split(qkv_w, 768, dim=1)
        q_b, k_b, v_b = torch.split(qkv_b, 768, dim=0)

        tb.mutihead_atten.w_query.weight = assign(tb.mutihead_atten.w_query.weight, q_w.T)
        tb.mutihead_atten.w_key.weight   = assign(tb.mutihead_atten.w_key.weight,   k_w.T)
        tb.mutihead_atten.w_value.weight = assign(tb.mutihead_atten.w_value.weight, v_w.T)
        tb.mutihead_atten.w_query.bias   = assign(tb.mutihead_atten.w_query.bias,   q_b)
        tb.mutihead_atten.w_key.bias     = assign(tb.mutihead_atten.w_key.bias,     k_b)
        tb.mutihead_atten.w_value.bias   = assign(tb.mutihead_atten.w_value.bias,   v_b)

        # Attention output projection
        tb.mutihead_atten.out_proj.weight = assign(
            tb.mutihead_atten.out_proj.weight, w[f"{p}.attn.c_proj.weight"].T)
        tb.mutihead_atten.out_proj.bias   = assign(
            tb.mutihead_atten.out_proj.bias,   w[f"{p}.attn.c_proj.bias"])

        # FeedForward
        tb.feed_forward.layer[0].weight = assign(
            tb.feed_forward.layer[0].weight, w[f"{p}.mlp.c_fc.weight"].T)
        tb.feed_forward.layer[0].bias   = assign(
            tb.feed_forward.layer[0].bias,   w[f"{p}.mlp.c_fc.bias"])
        tb.feed_forward.layer[2].weight = assign(
            tb.feed_forward.layer[2].weight, w[f"{p}.mlp.c_proj.weight"].T)
        tb.feed_forward.layer[2].bias   = assign(
            tb.feed_forward.layer[2].bias,   w[f"{p}.mlp.c_proj.bias"])

    # Final LayerNorm
    model.final_norm.scale = assign(model.final_norm.scale, w["ln_f.weight"])
    model.final_norm.shift = assign(model.final_norm.shift, w["ln_f.bias"])

    # Output head (weight tying)
    model.out_head.weight  = assign(model.out_head.weight,  w["wte.weight"])

    print("✅ Weights loaded successfully!")

In [4]:
from GPT_Model import *

In [5]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.0,       # set to 0 for inference
    "qkv_bias": True        # GPT-2 uses bias
}



In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gpt_model = GPT(cfg)        # must recreate after fixing cfg!
gpt_model.eval()

load_hf_into_my_gpt(gpt_model, hf_weights)
gpt_model.to(device)



✅ Weights loaded successfully!


GPT(
  (token_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (tranformer_block): Sequential(
    (0): Transfomers(
      (layernorm1): LayerNorm()
      (layernorm2): LayerNorm()
      (mutihead_atten): MultiHeadAttention(
        (w_query): Linear(in_features=768, out_features=768, bias=True)
        (w_key): Linear(in_features=768, out_features=768, bias=True)
        (w_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (dropout): Dropout(p=0.1, inplace=False)
      (feed_forward): FeedForward(
        (layer): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (1): Transfomers(
      (layernorm1): LayerNorm()
      (layernorm2): Laye

In [7]:
def Predict(gpt_model=gpt_model,prompt="",next_new_token=5,temp=.7,topk=4):
  #temp+topk implemention to find next token generation
  tokenid=generate_text(
    model=gpt_model,
    ip_token_id = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device),
    max_new_tokens=next_new_token,
    context_size=int(cfg["context_len"]/4),
    temp=temp,
    topk=topk
  )
  return (token_id_to_text(tokenid,tokenizer))

# APPLY KV(key value cache to GPT Arch)

In [8]:
import torch
import torch.nn as nn
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.epsilon = 1e-5
        self.shift = nn.Parameter(torch.zeros(emb_dim))
        self.scale = nn.Parameter(torch.ones(emb_dim))

    def forward(self, input):
        mean = input.mean(dim=-1, keepdim=True)
        var  = input.var(dim=-1, keepdim=True, unbiased=False)
        input_norm = (input - mean) / torch.sqrt(var + self.epsilon)
        return input_norm * self.scale + self.shift


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads, dropout, qkv_bias=False):
        super().__init__()
        self.d_in       = d_in
        self.d_out      = d_out
        self.num_head   = num_heads
        self.head_dim   = d_out // num_heads

        self.w_query  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_key    = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_value  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout  = nn.Dropout(dropout)

        # Causal mask (only used during prefill or non-cached forward passes)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x, past_kv=None, use_cache=False):
        """
        Args:
            x         : (batch, seq_len, d_in)  — seq_len == 1 during decode steps
            past_kv   : None  |  (past_key, past_value)
                          past_key/value shape: (batch, num_heads, past_len, head_dim)
            use_cache : whether to return the updated (key, value) for caching

        Returns:
            context_vec : (batch, seq_len, d_out)
            present_kv  : (key, value) tuple if use_cache else None
        """
        batch_size, num_token, _ = x.shape

        # Project to Q, K, V and reshape to (batch, heads, seq_len, head_dim)
        keys   = self.w_key(x)  .view(batch_size, num_token, self.num_head, self.head_dim).transpose(1, 2)
        querys = self.w_query(x).view(batch_size, num_token, self.num_head, self.head_dim).transpose(1, 2)
        values = self.w_value(x).view(batch_size, num_token, self.num_head, self.head_dim).transpose(1, 2)

        # ── KV Cache: concatenate new K,V with cached past K,V ──────────────
        if past_kv is not None:
            past_key, past_value = past_kv
            keys   = torch.cat([past_key,   keys],   dim=2)   # (B, H, past+cur, head_dim)
            values = torch.cat([past_value, values], dim=2)

        present_kv = (keys, values) if use_cache else None

        # total sequence length after concatenation
        total_len = keys.shape[2]   # past_len + num_token

        # ── Attention scores ─────────────────────────────────────────────────
        atten_scores = querys @ keys.transpose(2, 3)   # (B, H, num_token, total_len)

        # ── Causal mask ──────────────────────────────────────────────────────
        # During decode (num_token == 1) no masking is needed: the single query
        # is always the last token and can attend to everything before it.
        # During prefill we apply the standard causal mask.
        if num_token > 1:
            # When past tokens are present (e.g. chunked prefill), the new
            # tokens can attend freely to the past, but must be causally masked
            # among themselves.
            past_len = total_len - num_token
            # Start with all-False (no masking)
            mask = torch.zeros(num_token, total_len, device=x.device, dtype=torch.bool)
            # Apply upper-triangular mask only over the new-token columns
            causal = torch.triu(
                torch.ones(num_token, num_token, device=x.device, dtype=torch.bool),
                diagonal=1
            )
            mask[:, past_len:] = causal
            atten_scores = atten_scores.masked_fill(mask.unsqueeze(0).unsqueeze(0), -torch.inf)

        # Scale and softmax
        attn_weight = torch.softmax(atten_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weight = self.dropout(attn_weight)

        # Weighted sum then reshape back
        context_vec = (attn_weight @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(batch_size, num_token, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec, present_kv


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layer(x)


cfg = GPT_2_config = {
    "vocab":       50257,
    "context_len": 1024,
    "emb_dim":     768,
    "n_head":      12,
    "n_layer":     12,
    "dropout":     0.1,
    "qkv_bias":    True,
}


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layernorm1    = LayerNorm(cfg["emb_dim"])
        self.layernorm2    = LayerNorm(cfg["emb_dim"])
        self.mutihead_atten = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_len"],
            num_heads=cfg["n_head"],
            dropout=cfg["dropout"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.dropout      = nn.Dropout(cfg["dropout"])
        self.feed_forward = FeedForward(cfg)

    def forward(self, x, past_kv=None, use_cache=False):
        """
        Returns:
            x          : (batch, seq_len, emb_dim)
            present_kv : (K, V) if use_cache else None
        """
        # --- Self-attention with residual ---
        shortcut = x
        x, present_kv = self.mutihead_atten(
            self.layernorm1(x),
            past_kv=past_kv,
            use_cache=use_cache,
        )
        x = self.dropout(x) + shortcut

        # --- Feed-forward with residual ---
        shortcut = x
        x = self.dropout(self.feed_forward(self.layernorm2(x))) + shortcut

        return x, present_kv


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.token_emb  = nn.Embedding(cfg["vocab"],       cfg["emb_dim"])
        self.pos_emb    = nn.Embedding(cfg["context_len"], cfg["emb_dim"])
        self.dropout    = nn.Dropout(cfg["dropout"])

        # ── ModuleList instead of Sequential so we can pass past_kv per layer ──
        self.transformer_blocks = nn.ModuleList(
            [TransformerBlock(cfg) for _ in range(cfg["n_layer"])]
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head   = nn.Linear(cfg["emb_dim"], cfg["vocab"])

    def forward(self, ip_batch, past_kvs=None, use_cache=False):
        """
        Args:
            ip_batch  : (batch, seq_len)  — during decode this is (batch, 1)
            past_kvs  : None  |  list of (K, V) tuples, one per layer
            use_cache : return new per-layer (K, V) tuples

        Returns:
            logit      : (batch, seq_len, vocab)
            present_kvs: list[(K, V)] per layer if use_cache else None
        """
        batch, seq_len = ip_batch.shape

        # Positional offset: if KV cache already holds past tokens, start positions
        # from past_len so embeddings stay consistent across prefill and decode.
        past_len = past_kvs[0][0].shape[2] if past_kvs is not None else 0

        token_embd = self.token_emb(ip_batch)
        pos_ids    = torch.arange(past_len, past_len + seq_len, device=ip_batch.device)
        pos_embd   = self.pos_emb(pos_ids)

        x = self.dropout(token_embd + pos_embd)

        present_kvs = []
        for i, block in enumerate(self.transformer_blocks):
            layer_past = past_kvs[i] if past_kvs is not None else None
            x, present_kv = block(x, past_kv=layer_past, use_cache=use_cache)
            if use_cache:
                present_kvs.append(present_kv)

        x     = self.final_norm(x)
        logit = self.out_head(x)

        return logit, (present_kvs if use_cache else None)


# ── Sampling helper ───────────────────────────────────────────────────────────

def _sample_next(logit, temp, topk):
    """Apply temperature + top-k then sample one token. logit: (batch, vocab)"""
    if temp > 0:
        logit = logit / temp

    if topk is not None:
        top_logit, _ = torch.topk(logit, topk)
        min_topk = top_logit[:, -1].unsqueeze(-1)
        logit = torch.where(logit < min_topk,
                            torch.full_like(logit, float("-inf")),
                            logit)

    prob = torch.softmax(logit, dim=-1)
    return torch.multinomial(prob, num_samples=1)   # (batch, 1)


# ── KV-cache–aware text generation ───────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def generate_text_KV(model, ip_token_id, max_new_tokens, context_size, temp, topk=None):
    """
    Fast autoregressive generation using KV cache.

    Workflow
    --------
    1. **Prefill**  — run the full prompt through the model once, storing K,V
                      for every layer.
    2. **Decode**   — for each new token, forward only the single new token;
                      the attention layer reads from the cache instead of
                      reprocessing the entire sequence.

    This reduces per-step compute from O(n²) to O(n) in sequence length.
    """
    model.eval()

    # ── Step 1: Prefill ──────────────────────────────────────────────────────
    # Clamp the prompt to the model's context window
    context_ip = ip_token_id[:, -context_size:]

    with torch.no_grad():
        logit, past_kvs = model(context_ip, past_kvs=None, use_cache=True)

    # Sample the first new token from the last position of the prefill output
    idx_next    = _sample_next(logit[:, -1, :], temp, topk)
    ip_token_id = torch.cat([ip_token_id, idx_next], dim=1)

    # ── Step 2: Decode (one token at a time) ─────────────────────────────────
    for _ in range(max_new_tokens - 1):
        # Guard: never exceed context_size total
        total_len = past_kvs[0][0].shape[2]   # cached sequence length
        if total_len >= context_size:
            # Slide the cache window: drop the oldest token from every layer
            past_kvs = [
                (k[:, :, 1:, :], v[:, :, 1:, :])
                for k, v in past_kvs
            ]

        with torch.no_grad():
            # Forward only the single new token (seq_len = 1)
            logit, past_kvs = model(idx_next, past_kvs=past_kvs, use_cache=True)

        idx_next    = _sample_next(logit[:, -1, :], temp, topk)
        ip_token_id = torch.cat([ip_token_id, idx_next], dim=1)

    return ip_token_id


# ── Token ↔ text helpers (unchanged) ─────────────────────────────────────────

def text_to_token_id(text, tokenizer=tokenizer):
    text_batch = [torch.tensor(tokenizer.encode(t)) for t in text]
    return torch.stack(text_batch)


def token_id_to_text(token_id, tokenizer=tokenizer):
    return [tokenizer.decode(token_id[i].squeeze(0).tolist()) for i in range(len(token_id))]


if __name__ == "__main__":
    model = GPT(GPT_2_config).to(device)


In [9]:
def Predict_KV_Cache(gpt_model=gpt_model,prompt="",next_new_token=5,temp=.7,topk=4):
  token_ids = text_to_token_id(prompt).to(device)

  out = generate_text_KV(
        model=model,
        ip_token_id=token_ids,
        max_new_tokens=next_new_token,
        context_size=GPT_2_config["context_len"],
        temp=temp,
        topk=topk,
    )
  return (token_id_to_text(out)[0])

In [10]:
# Test inference without K-V cache
import time
start_time=time.time()
out=Predict(prompt="Every step move you",next_new_token=7,temp=1,topk=3)
end_time=time.time()
print(f"Total time take by Test inference without K-V cache:{out}, time: {end_time-start_time}")

Total time take by Test inference without K-V cache:['Every step move you can take is a very good thing'], time: 0.73807692527771


In [11]:
# Test inference With K-V cache
import time
start_time=time.time()
out=Predict_KV_Cache(prompt="Every step move you",next_new_token=7,temp=1,topk=3)
end_time=time.time()
print(f"Total time take by Test inference with K-V cache:{out},time {end_time-start_time}")

Total time take by Test inference with K-V cache:E 365serving rodents statedisplayText eviction helmets,time 0.33948183059692383
